In [ ]:
import sys
import os

# detect the environment
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# configure the paths
if IN_COLAB:
    print("Running in Google Colab. Setting up GitHub repo...")
    REPO_URL = "https://github.com/JayC-SF/COMP-432-Project.git"
    REPO_DIR = "/content/COMP-432-Project"
    
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL}
        
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # change the working directory
    os.chdir(REPO_DIR)
else:
    print("Running locally. Setting up relative paths...")
    # move up only if base directory is at notebooks
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
        print(f"Working directory changed to: {os.getcwd()}")
    
    # add working dir to sys path
    if os.getcwd() not in sys.path:
        sys.path.append(os.getcwd())
        
    %load_ext autoreload
    %autoreload 2

from src import preprocess_data as prepd
from src.models.NN import AudioCLSTM
import src.variables as v
import numpy as np
import torch
import secrets
from src.train.orchestrator import Orchestrator
from src.utils.hardware import get_device
from src.utils.seed import set_seed
from src.datasets import ICSD_MelSpectogram
from torch.utils.data import DataLoader
import copy
set_seed(v.SEED)

Running locally. Setting up relative paths...
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Seed set to: 42


Download the mel spectogram `.npz` dataset.

In [3]:
prepd.download_google_file(v.MEL_SPECTOGRAM_NPZ_FILE_PATH, v.MEL_SPECTOGRAM_NPZ_GID)

data/mel_spectogram_audio_length_adjusted.npz does not exists, proceeding to download...


Downloading...
From (original): https://drive.google.com/uc?id=1Bjf5uWVFMzlDm77xxZAbbIVInit6mZQK
From (redirected): https://drive.google.com/uc?id=1Bjf5uWVFMzlDm77xxZAbbIVInit6mZQK&confirm=t&uuid=e3512e29-8bf7-4179-a752-3630e2a1ae3b
To: /Users/studybuggy/Desktop/concordia-code/9.winter2026/COMP432/COMP-432-Project/data/mel_spectogram_audio_length_adjusted.npz
100%|██████████| 2.22G/2.22G [00:55<00:00, 39.8MB/s]


In [4]:
data = np.load(v.MEL_SPECTOGRAM_NPZ_FILE_PATH)
train_ds = ICSD_MelSpectogram(data['X_train'], data['y_train'])
val_ds = ICSD_MelSpectogram(data['X_val'], data['y_val'])
test_ds = ICSD_MelSpectogram(data['X_test'], data['y_test'])

We load the dataset into their respective splits

In [ ]:
set_seed(v.SEED)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False)

✅ Seed set to: 42


In [ ]:
set_seed(v.SEED)
untrained_classic_cnn_model = AudioCLSTM(2)

✅ Seed set to: 42


In [ ]:
set_seed(v.SEED)
# use output 2 classes so we can use standard multi class logic for training orchestrator
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 5e-4
PATIENCE = 15
SAVE_PATH = v.RUNS_PATH/f"CLSTM_{secrets.token_hex(3)}" # put random hash in case of rerunning and overwriting by accident
MAX_EPOCHS = 500
model = copy.deepcopy(untrained_classic_cnn_model)
DEVICE = get_device()
optimizer = torch.optim.AdamW(
    model.parameters(),
    weight_decay = WEIGHT_DECAY,
    lr=LEARNING_RATE
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    patience=5, 
    factor=0.1, 
)
criterion = torch.nn.CrossEntropyLoss()

orchestrator = Orchestrator(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    patience=PATIENCE,
    save_path=SAVE_PATH,
    scheduler=scheduler,
    max_epochs=MAX_EPOCHS,
    classes=v.CLASSES
)

orchestrator.train()

✅ Seed set to: 42
Running with device:cpu
---- Starting Epoch 1 ----


KeyboardInterrupt: 